# 🏏 Rajasthan Royals Hackathon - Top 40 Uncapped Indian Players
## Elite Cricket Intelligence Framework

**Goal**: Identify uncapped Indian players most likely to receive international caps using:
- Capping Likelihood Engine (CLE) - 8-dimensional custom scoring
- Trajectory Modelling - Future performance prediction
- Spatial Cricket Intelligence - Role-aware evaluation
- Player Archetype Clustering - IPL-style talent ID
- WTO Intelligence - Wicket-Taking Opportunity analysis
- Advanced ML Pipeline - LightGBM + XGBoost ensemble

---

## 📦 Section 1: Environment Setup & Imports

In [20]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
#import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ML Libraries
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, precision_recall_curve, auc
from sklearn.model_selection import train_test_split, StratifiedKFold
import lightgbm as lgb
import xgboost as xgb

# Statistics & Math
from scipy import stats
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_similarity

# Datetime
from datetime import datetime, timedelta

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 📂 Section 2: Data Loading

In [21]:
# File paths
DATA_PATH = r'e:\rajaistan_unsto[\2 round\drive-download-20251203T140652Z-1-001'

print("Loading datasets...")

# Load all CSV files
ball_by_ball = pd.read_csv(f'{DATA_PATH}\\ball_by_ball.csv')
matches = pd.read_csv(f'{DATA_PATH}\\matches.csv')
players = pd.read_csv(f'{DATA_PATH}\\players.csv')
playing_eleven = pd.read_csv(f'{DATA_PATH}\\playing_eleven.csv')
competitions = pd.read_csv(f'{DATA_PATH}\\competitions.csv')
player_team_comp = pd.read_csv(f'{DATA_PATH}\\player_team_competition_mapping.csv')

print(f"\n✅ Data loaded successfully!")
print(f"\n📊 Dataset Sizes:")
print(f"  - ball_by_ball: {len(ball_by_ball):,} rows")
print(f"  - matches: {len(matches):,} rows")
print(f"  - players: {len(players):,} rows")
print(f"  - playing_eleven: {len(playing_eleven):,} rows")
print(f"  - competitions: {len(competitions):,} rows")
print(f"  - player_team_comp: {len(player_team_comp):,} rows")

# Display column names
print(f"\n📋 Ball-by-ball columns: {ball_by_ball.columns.tolist()[:10]}...")

Loading datasets...

✅ Data loaded successfully!

📊 Dataset Sizes:
  - ball_by_ball: 602,992 rows
  - matches: 2,714 rows
  - players: 5,468 rows
  - playing_eleven: 61,478 rows
  - competitions: 173 rows
  - player_team_comp: 18,701 rows

📋 Ball-by-ball columns: ['match_id', 'innings', 'overs', 'over_num', 'innings_ball_num', 'over_ball_num', 'batting_team_id', 'batting_team_name', 'bowling_team_id', 'bowling_team_name']...


## 🗺️ Section 3: Spatial Mapping Functions (Reference Doc)

Implementing exact pitch zone, wagon wheel, and shot zone mappings from the reference document.

In [22]:
# ==================== PITCH MAP ZONES (1-63) ====================

# Length buckets - exact zone sets from reference doc
LENGTH_MAP = {
    'yorker': set(range(10, 19)),           # Zones 10-18
    'full': set(range(19, 28)),             # 2m-4m length
    'good': set(range(28, 37)),             # 4m-6m (good length)
    'back_of_length': set(range(37, 46)),   # Back of length
    'short': set(range(46, 64))             # Short/bouncer (46-63)
}

# Line buckets - exact zone sets from reference doc
LINE_MAP = {
    'off_stump': {4, 13, 22, 31, 40, 49, 58},
    'outside_off': {3, 12, 21, 30, 39, 48, 57},
    'deep_outside_off': {2, 11, 20, 29, 38, 47, 56},
    'deepest_outside_off': {1, 10, 19, 28, 37, 46, 55},
    'middle_stump': {5, 14, 23, 32, 41, 50, 59},
    'leg_stump': {6, 15, 24, 33, 42, 51, 60},
    'outside_leg': {7, 16, 25, 34, 43, 52, 61},
    'deep_outside_leg': {8, 17, 26, 35, 44, 53, 62},
    'deepest_outside_leg': {9, 18, 27, 36, 45, 54, 63}
}

def get_length_from_zone(zone):
    """Map pitch zone (1-63) to length category."""
    if pd.isna(zone):
        return 'unknown'
    zone = int(zone)
    for length, zones in LENGTH_MAP.items():
        if zone in zones:
            return length
    return 'unknown'

def get_line_from_zone(zone):
    """Map pitch zone (1-63) to line category."""
    if pd.isna(zone):
        return 'unknown'
    zone = int(zone)
    for line, zones in LINE_MAP.items():
        if zone in zones:
            return line
    return 'unknown'

def get_length_line_combo(zone):
    """Get combined length-line description."""
    if pd.isna(zone):
        return 'unknown'
    length = get_length_from_zone(zone)
    line = get_line_from_zone(zone)
    return f"{length}_{line}"

print("✅ Pitch zone mapping functions created")
print(f"\nExample: Zone 30 → Length: {get_length_from_zone(30)}, Line: {get_line_from_zone(30)}")
print(f"Example: Zone 15 → Length: {get_length_from_zone(15)}, Line: {get_line_from_zone(15)}")

✅ Pitch zone mapping functions created

Example: Zone 30 → Length: good, Line: outside_off
Example: Zone 15 → Length: yorker, Line: leg_stump


In [23]:
# ==================== WAGON WHEEL ZONES (1-24) ====================

# Approximate wagon wheel sector mapping (clockwise from long-off)
WAGON_WHEEL_SECTORS = {
    1: 'long_off', 2: 'extra_cover', 3: 'cover', 4: 'point',
    5: 'backward_point', 6: 'third_man', 7: 'fine_leg', 8: 'deep_fine_leg',
    9: 'deep_square_leg', 10: 'square_leg', 11: 'mid_wicket', 12: 'deep_mid_wicket',
    13: 'long_on', 14: 'straight', 15: 'mid_on', 16: 'cow_corner',
    17: 'deep_cover', 18: 'sweeper', 19: 'backward_square', 20: 'leg_side',
    21: 'off_side', 22: 'behind_wicket', 23: 'in_front', 24: 'aerial'
}

def get_wagon_sector(zone):
    """Map wagon wheel zone to field sector."""
    if pd.isna(zone):
        return 'unknown'
    zone = int(zone)
    return WAGON_WHEEL_SECTORS.get(zone, 'unknown')

print("✅ Wagon wheel mapping functions created")

✅ Wagon wheel mapping functions created


In [24]:
# ==================== SHOT ZONES (1-42) ====================

def get_shot_height_category(zone):
    """Categorize shot zone by height (low/medium/loft)."""
    if pd.isna(zone):
        return 'unknown'
    zone = int(zone)
    height_y = ((zone - 1) // 7) + 1
    
    if height_y in [1, 2]:
        return 'ground_stroke'
    elif height_y in [3, 4]:
        return 'aerial_controlled'
    else:
        return 'lofted'

def get_shot_line_category(zone):
    """Categorize shot zone by line (left/center/right)."""
    if pd.isna(zone):
        return 'unknown'
    zone = int(zone)
    line_x = ((zone - 1) % 7) + 1
    
    if line_x in [1, 2]:
        return 'off_side'
    elif line_x in [3, 4, 5]:
        return 'straight'
    else:
        return 'leg_side'

print("✅ Shot zone mapping functions created")

✅ Shot zone mapping functions created


## 🔍 Section 4: Identify Capped vs Uncapped Players

In [25]:
# Identify international competitions
print("Identifying international competitions...\n")
print("All competitions:")
print(competitions[['id', 'name']].drop_duplicates())

# Keywords indicating international matches
intl_keywords = ['T20I', 'ODI', 'Test', 'International', 'World Cup', 'Champions Trophy']

# Identify international competition IDs
intl_comp_ids = competitions[
    competitions['name'].str.contains('|'.join(intl_keywords), case=False, na=False)
]['id'].unique()

print(f"\n🌍 International competition IDs: {intl_comp_ids}")

Identifying international competitions...

All competitions:
       id                                          name
0    3289                    Asia Cup Rising Stars 2025
1    3288                 Pakistan T20I Tri-Series 2025
2    3282   West Indies in New Zealand T20I Series 2025
3    3029              Chiripal T20 Premier League 2025
4    3089                Delhi Mens Premier League 2025
..    ...                                           ...
168  3020      Bangladesh in Sri Lanka T20I Series 2025
169  2874     Australia in West Indies T20I Series 2025
170  2912    South Africa in Australia T20I Series 2025
171  3074       Pakistan in Bangladesh T20I Series 2025
172  2906  West Indies in South Africa T20I Series 2026

[173 rows x 2 columns]

🌍 International competition IDs: [3288 3282 3126 3128 3211 3223 3226 3228 3241 3197 2535 2678 1223 1146
 1224 1115 1194 1156 1171 1144 1161 1190 1265 1361 1360 1366 1494 1331
 1435 1584 1345 1611 1335 1517 1595 1452 1537 1603 1499 1439 1599 1

In [26]:
# Identify ALL capped players (any era)
print("Identifying capped players...\n")

# Get players who played in international competitions
capped_players = player_team_comp[
    player_team_comp['comp_id'].isin(intl_comp_ids)
]['player_id'].unique()

print(f"✅ Found {len(capped_players)} capped players in dataset")

# Mark all players as capped or uncapped
players['is_capped'] = players['player_id'].isin(capped_players).astype(int)

print(f"\n📊 Capped vs Uncapped:")
print(players['is_capped'].value_counts())

# Filter for Indian players
if 'nationality' in players.columns:
    indian_players = players[players['nationality'].str.contains('India', case=False, na=False)]
    print(f"\n🇮🇳 Indian players: {len(indian_players)}")
    print(f"  - Capped: {indian_players['is_capped'].sum()}")
    print(f"  - Uncapped: {len(indian_players) - indian_players['is_capped'].sum()}")
else:
    print("\n⚠️ No nationality column found")

Identifying capped players...

✅ Found 843 capped players in dataset

📊 Capped vs Uncapped:
is_capped
0    4733
1     735
Name: count, dtype: int64

🇮🇳 Indian players: 4089
  - Capped: 285
  - Uncapped: 3804


## 🔗 Section 5: Data Integration & Master Dataset Creation

In [27]:
print("Creating master dataset...\n")

# Step 1: Merge ball_by_ball with matches
print("Step 1: Merging ball_by_ball with matches...")
bbb_merged = ball_by_ball.merge(
    matches[['match_id', 'comp_id', 'match_date', 'venue_name', 'win_team_id']], 
    on='match_id', 
    how='left'
)
print(f"  ✓ Merged: {len(bbb_merged):,} rows")

# Step 2: Add competition info
print("Step 2: Adding competition info...")
bbb_merged = bbb_merged.merge(
    competitions[['id', 'name']].rename(columns={'id': 'comp_id', 'name': 'comp_name'}), 
    on='comp_id', 
    how='left'
)
print(f"  ✓ Competition names added")

# Step 3: Apply spatial mappings (CORRECTED COLUMN NAME: picth_map_zone)
print("Step 3: Applying spatial mappings...")
bbb_merged['pitch_length'] = bbb_merged['picth_map_zone'].apply(get_length_from_zone)
bbb_merged['pitch_line'] = bbb_merged['picth_map_zone'].apply(get_line_from_zone)
bbb_merged['pitch_length_line'] = bbb_merged['picth_map_zone'].apply(get_length_line_combo)
bbb_merged['wagon_sector'] = bbb_merged['wagon_wheel_zone'].apply(get_wagon_sector)
bbb_merged['shot_height'] = bbb_merged['shot_zone'].apply(get_shot_height_category)
bbb_merged['shot_line'] = bbb_merged['shot_zone'].apply(get_shot_line_category)
print(f"  ✓ Spatial columns added")

# Step 4: Add phase classification
print("Step 4: Classifying match phases...")
bbb_merged['phase'] = pd.cut(
    bbb_merged['over_num'], 
    bins=[0, 6, 15, 100], 
    labels=['powerplay', 'middle', 'death'],
    include_lowest=True
)
print(f"  ✓ Phase classification complete")

print(f"\n✅ Master dataset created with {len(bbb_merged):,} rows")
print(f"\nNew columns: pitch_length, pitch_line, wagon_sector, shot_height, shot_line, phase")

# Display sample
bbb_merged[[
    'striker_name', 'bowler_name', 'over_num', 'phase', 
    'pitch_length', 'pitch_line', 'wagon_sector', 'runs'
]].head(10)

Creating master dataset...

Step 1: Merging ball_by_ball with matches...
  ✓ Merged: 602,992 rows
Step 2: Adding competition info...
  ✓ Competition names added
Step 3: Applying spatial mappings...
  ✓ Spatial columns added
Step 4: Classifying match phases...
  ✓ Phase classification complete

✅ Master dataset created with 602,992 rows

New columns: pitch_length, pitch_line, wagon_sector, shot_height, shot_line, phase


,striker_name,bowler_name,over_num,phase,pitch_length,pitch_line,wagon_sector,runs
0,Brendan Taylor,Faheem Ashraf,4,powerplay,good,deep_outside_off,straight,0
1,Brendan Taylor,Faheem Ashraf,4,powerplay,short,deep_outside_off,cow_corner,0
2,Brendan Taylor,Naseem Shah,3,powerplay,back_of_length,deep_outside_off,fine_leg,0
3,Brendan Taylor,Naseem Shah,3,powerplay,short,off_stump,deep_cover,4
4,Brendan Taylor,Naseem Shah,3,powerplay,back_of_length,deep_outside_off,deep_square_leg,0
5,Brendan Taylor,Naseem Shah,3,powerplay,back_of_length,outside_off,fine_leg,0
6,Brendan Taylor,Naseem Shah,3,powerplay,back_of_length,outside_off,third_man,0
7,Brendan Taylor,Naseem Shah,3,powerplay,back_of_length,deep_outside_off,deep_cover,4
8,Ryan Burl,Mohammad Nawaz,7,middle,good,outside_leg,backward_square,1
9,Ryan Burl,Mohammad Nawaz,11,middle,good,outside_leg,third_man,0


## 💾 Section 6: Save Progress

In [28]:
# Save processed data
print("Saving processed data...")

bbb_merged.to_parquet(f'{DATA_PATH}\\..\\ball_by_ball_processed.parquet', index=False)
players.to_parquet(f'{DATA_PATH}\\..\\players_processed.parquet', index=False)

print("✅ Data saved successfully!")
print(f"  - ball_by_ball_processed.parquet")
print(f"  - players_processed.parquet")

Saving processed data...
✅ Data saved successfully!
  - ball_by_ball_processed.parquet
  - players_processed.parquet


---
## 📋 Progress Summary

**Completed**:
- ✅ Environment setup and library imports
- ✅ Data loading (6 CSV files)
- ✅ Spatial mapping functions:
  - Pitch zones (1-63) → length + line
  - Wagon wheel zones (1-24) → field sectors
  - Shot zones (1-42) → height + line
- ✅ Capped vs uncapped player identification
- ✅ Master dataset with spatial features
- ✅ Phase classification (powerplay/middle/death)

**Next**: Feature Engineering (84+ cricket intelligence features)
---